# CADET-GUI: staged characterization pipeline

This notebook walks the same staged, provenance-tracked characterization
pipeline the Lonza workflow-features alignment work targets: system
periphery -> column bed -> particle transport -> binding, each stage a
joint fit across as many experiments as you select, chained so each
stage's accepted result becomes the next stage's starting point.

**No experimental data is bundled with this notebook.** Import your own
CSV data per stage via that stage's own "Import CSV" control (two columns:
time, signal -- see `DataImportWidget`); an optional cell at the bottom
shows how to exercise the mechanics with synthetic data instead.

Unlike `examples/workbench.ipynb`, `CharacterizationWidget` is deliberately
**not** one of `WorkbenchWidget`'s default steps -- it overlaps
conceptually with `ParameterEstimationWidget` (both fit column/binding
parameters against experimental data), so it's reached only through a
notebook like this one, built the same way `examples/custom_sidebar.ipynb`
composes widgets under `SidebarShell`.

In [ ]:
import cadetgui.configuration_store as configuration_store
from cadetgui.widgets._sidebar_shell import SidebarShell
from cadetgui.widgets.composite import CharacterizationWidget, ConfigurationWidget, InstrumentWidget
from cadetgui.cadetprocessadapter import COLUMN_MODELS, BINDING_MODELS

## 1. Base configuration

One `InstrumentWidget` + `ConfigurationWidget` shared by every stage below
-- each stage's "Accept" writes fitted values back into *this same*
`config`/`instrument`, so state flows forward from stage to stage
automatically, in-memory, for the rest of this notebook session.

A column model with pores (here: LRMP) and a binding model with both a
characteristic charge and a capacity (here: SMA) are picked so every stage
below has something real to fit -- `CharacterizeBed`/`CharacterizeParticles`
need `bed_porosity`/`film_diffusion`, which a pore-less LRM column doesn't
have, and `CharacterizeAdsorptionParameters`/`CharacterizeCapacity` need
SMA-style attributes a `Linear` binding model doesn't have either.

In [ ]:
instrument = InstrumentWidget()
config = ConfigurationWidget(instrument=instrument)

instrument._use_lc_system_checkbox.value = True
# Periphery segments used by the two "Periphery: ..." stages below --
# unchecked units aren't part of the flow path, so switch these on to
# characterize them.
instrument._unit_checkboxes["tubing_pre_injection"].value = True
instrument._unit_checkboxes["tubing_detectors"].value = True
instrument._unit_checkboxes["mixer"].value = True

config._column_picker.value = COLUMN_MODELS["Lumped Rate Model With Pores (LRMP)"]
config._binding_picker.value = BINDING_MODELS["Steric Mass Action (SMA)"]

## 2. One widget per pipeline stage

Each `CharacterizationWidget` wraps one `CADETProcess.characterization`
class (see its own docstring for the full mapping). The two periphery
stages target different physical segments -- `tubing_pre_injection`
(before the column) and `tubing_detectors` (after it) -- so both a
pre-injection dead-volume run and a detector-tubing run can be
characterized independently, matching the reference workflow's own split
between upstream and downstream extra-column contributions.

`stage="pre_injection"` (`CharacterizePreInjection`) additionally fits the
mixer's volume jointly with the pre-injection tubing length -- a second,
narrower way to characterize the same upstream segment when the mixer's
own contribution matters too.

In [ ]:
periphery_pre_injection = CharacterizationWidget(
    "periphery", config=config, instrument=instrument, tubing_unit="tubing_pre_injection",
)
periphery_detectors = CharacterizationWidget(
    "periphery", config=config, instrument=instrument, tubing_unit="tubing_detectors",
)
pre_injection = CharacterizationWidget("pre_injection", config=config, instrument=instrument)
bed = CharacterizationWidget("bed", config=config)
particles = CharacterizationWidget("particles", config=config)
adsorption = CharacterizationWidget("adsorption", config=config)
capacity = CharacterizationWidget("capacity", config=config)

## 3. Navigate the pipeline

One pane per stage, in pipeline order. For each stage: import (or already
have imported) the relevant experimental dataset(s), pick which loaded
dataset(s) to fit jointly, click "Preview" to discover the available
simulated signals, adjust the bound-override fields if needed, pick an
optimizer (**U-NSGA-III is required, not Nelder-Mead, whenever more than
one dataset is selected** -- a joint multi-dataset fit is genuinely
multi-objective, one objective per dataset, which a single-objective
optimizer like Nelder-Mead cannot solve), Run, then Accept once you're
happy with the fit. Accept writes straight into `config`/`instrument`'s
own live form fields -- the next stage's pane already sees the update.

In [ ]:
shell = SidebarShell(
    [
        ("System", instrument.root),
        ("Configuration", config.root),
        ("Periphery: pre-injection", periphery_pre_injection.root),
        ("Periphery: detectors", periphery_detectors.root),
        ("Periphery: pre-injection + mixer", pre_injection.root),
        ("Bed", bed.root),
        ("Particles", particles.root),
        ("Adsorption", adsorption.root),
        ("Capacity", capacity.root),
    ]
)
shell.display()

## 4. Provenance

`config`'s own "Configuration" pane (above) has a Save button -- click it
after accepting a stage's fit to checkpoint that state to the local
configuration store, so it can be reloaded later (e.g. in a fresh kernel,
or by `ParameterEstimationWidget`'s own "Base process" picker) rather than
only living in this notebook's in-memory objects. Nothing here saves
automatically -- matching the rest of cadetgui's "never silently overwrite
without an explicit action" rule.

Everything saved so far, most recent first:

In [ ]:
for name, hash_ in configuration_store.list_store(store_dir=config.persistence.store_dir):
    print(f"{name}  ({hash_})")

## Optional: try the mechanics with synthetic data

No real experimental data required -- this builds one throwaway pulse-like
signal and drops it straight into the pre-injection periphery stage above,
so you can Preview/Run/Accept once and see the pipeline actually move,
before bringing in real data. Safe to skip entirely.

In [ ]:
import numpy as np
from cadetgui.widgets.composite.data_import import ExperimentalDataset

t = np.linspace(0, 20, 100)  # minutes
signal = np.exp(-((t - 8) ** 2) / 4)  # a synthetic pulse-response-shaped peak
periphery_pre_injection.data.datasets.append(
    ExperimentalDataset(label="synthetic pulse", time_min=t, signal=signal)
)
periphery_pre_injection._refresh_dataset_options()
print("Added -- open the 'Periphery: pre-injection' pane above, select it under Datasets, Preview, then Run.")